## IDP Phase: Extraction
Now that we have the blueprint and the project created, let's take a look at how we can use the BDA boto3 APIs to extract the data. To interact with the boto3 APIs, we will use the SageMaker notebooks.

Following are the steps we will demonstrate in this module:

* Set up the environment
* Upload the sample patient-records (ensure you downloaded the sample) to S3
* Invoke the async extraction API with a sample **simple_medical_record_inf.pdf**. Ensure extraction works
* Validate IDP phase **Classification** is working correctly by verifying that the document class is **Simple Patient Medical Chart**
* Invoke the API again with sample detailed medical record **detailed_medical_chart_inf.pdf**
* Verify that the document class is **Detailed Patient Medical Chart**
* Upload the extracted results of detailed medical chart to S3 bucket **idp-workshop-{account_number}-us-west-2/BDA-output**


### Step-1: Setup environment
* Setup the environment. Restart the kernel after this step

In [ ]:
%pip install --no-warn-conflicts "boto3>=1.37.6" itables==2.2.4 PyPDF2==3.0.1 --upgrade -qq

In [ ]:
%load_ext autoreload
%autoreload 1

### Step-2: Initialize Parameters
* Before we get to the part where we invoke BDA with our sample artifacts, let's setup some parameters and configuration that will be used throughout this notebook.

In [ ]:
import boto3
import json
from IPython.display import JSON, IFrame
import sagemaker
import pandas as pd
from utils import display_functions, helper_functions
from pathlib import Path
import os

session = sagemaker.Session()
# Remove this bucket if not needed - Vinay
default_bucket = session.default_bucket()
current_region = boto3.session.Session().region_name

sts_client = boto3.client('sts')
account_number = sts_client.get_caller_identity()['Account']
# Define bucket and folder paths
bucket_name = f"idp-workshop-{account_number}-us-west-2"
default_bucket = bucket_name
input_folder = "BDA-input" # Store  input documents
output_folder = "BDA-output"  # Write extracted data

# Initialize Bedrock Data Automation client
bda_client = boto3.client('bedrock-data-automation')
bda_runtime_client = boto3.client('bedrock-data-automation-runtime')
s3_client = boto3.client('s3')

bda_s3_input_location = f's3://{bucket_name}/{input_folder}'
bda_s3_output_location = f's3://{bucket_name}/{output_folder}'
print(bda_s3_input_location)

### Step-3: Prepare sample document
* Lets upload the sample document from local folder **patient-records/inference-samples** to S3


In [ ]:
local_download_path = '../patient-records/inference-samples'

# Upload simple_medical_chart_inf.pdf to S3
simple_file_name = 'simple_medical_chart_inf.pdf'
simple_local_file_path = os.path.join(local_download_path, simple_file_name)
simple_document_s3_uri = f'{bda_s3_input_location}/{simple_file_name}'

target_s3_bucket, target_s3_key =  helper_functions.get_bucket_and_key(simple_document_s3_uri)
s3_client.upload_file(simple_local_file_path, target_s3_bucket, target_s3_key)

print('#####################################')
print(f"File local path: {simple_local_file_path}")
print(f"Uploaded file to S3: {target_s3_key}")
print(f"simple_document_s3_uri: {simple_document_s3_uri}")
print('#####################################')


# Upload detailed_medical_chart_inf.pdf to S3
detailed_file_name = 'detailed_medical_chart_inf.pdf'
detailed_local_file_path = os.path.join(local_download_path, detailed_file_name)
detailed_document_s3_uri = f'{bda_s3_input_location}/{detailed_file_name}'

target_s3_bucket, target_s3_key =  helper_functions.get_bucket_and_key(detailed_document_s3_uri)
s3_client.upload_file(detailed_local_file_path, target_s3_bucket, target_s3_key)

print(f"File local path: {detailed_local_file_path}")
print(f"Uploaded file to S3: {target_s3_key}")
print(f"detailed_document_s3_uri: {detailed_document_s3_uri}")
print('#####################################')


### Step-4.1: View Simple Document

In [ ]:
IFrame(simple_local_file_path, width=600, height=400)

### Step-4.2: View Detailed Document

In [ ]:
IFrame(detailed_local_file_path, width=600, height=400)

### Step-5: Get Project ARN
* Go to the BDA project console and copy the Project ARN for **idp-bda-workshop-project**
* Also verify that the project is in good state
* Usually ARN is in the format of arn:aws:bedrock:region:account-id:data-automation-project/3awsdddd
* **Note:** update below line with the ARN from the project in your account

In [ ]:
project_arn = 'arn:aws:bedrock:us-west-2:462422080976:data-automation-project/01eff3d76996'

### Step-6: Async Extraction for Simple Medical Chart
* Invoke Data Automation Async API for extraction of **simple_medical_chart_inf.pdf**
* This step returns the invocation ARN
* Poll invocation status
* Validate Classification
* View Matched Blueprints, Output Summary

### Step-6.1: Extraction
* Invoke Data Automation Async API for extraction of **simple_medical_chart_inf.pdf**
* This step returns the invocation ARN

In [ ]:
response = bda_runtime_client.invoke_data_automation_async(
    inputConfiguration={
        's3Uri': simple_document_s3_uri
    },
    outputConfiguration={
        's3Uri': bda_s3_output_location
    },
    dataAutomationConfiguration={
        'dataAutomationProjectArn': project_arn,
        'stage': 'LIVE'
    }, 
    dataAutomationProfileArn = f'arn:aws:bedrock:{current_region}:{account_number}:data-automation-profile/us.data-automation-v1'
)

invocationArn = response['invocationArn']
print(f'Invoked data automation job with invocation arn {invocationArn}')

### Step-6.2: Poll for Invocation Status
* Since invocation is an async step, we will poll for status to be completed
* The invocation job status moves from `Created` to `InProgress` and finally to `Success` when the job completes successfully
* If the job encounters an error the final status is either `ServiceError` or `ClientError` with error details
* Run this step couple of time till the status is `Success`

In [ ]:
status_response = helper_functions.wait_for_completion(
            client=bda_client,
            get_status_function=bda_runtime_client.get_data_automation_status,
            status_kwargs={'invocationArn': invocationArn},
            completion_states=['Success'],
            error_states=['ClientError', 'ServiceError'],
            status_path_in_response='status',
            max_iterations=15,
            delay=30
)
if status_response['status'] == 'Success':
    job_metadata_s3_location = status_response['outputConfiguration']['s3Uri']
else:
    raise Exception(f'Invocation Job Error, error_type={status_response["error_type"]},error_message={status_response["error_message"]}')

### Step-6.3: Validate Classification
* Retrieve the job metadata. The Job metadata contains the S3 uri's for the standard output,custom output and the status of custom output. The custom output status could be either of `MATCH` or `NO_MATCH`.
* `MATCH` indicates BDA was able to find a matching blueprint for the specific segment from the list of blueprint we associated with the project.


In [ ]:
job_metadata = json.loads(helper_functions.read_s3_object(job_metadata_s3_location))

job_metadata_table = pd.DataFrame(job_metadata['output_metadata'][0]['segment_metadata']).fillna('')
job_metadata_table.index.name='Segment Index'
job_metadata_json = JSON(job_metadata, root="job_metadata", expanded=True)
# Display the widget
display_functions.display_multiple(
    [display_functions.get_view(job_metadata_table), display_functions.get_view(job_metadata_json)], 
    ["Table View", "Raw JSON"])

### Step-6.4: View  and Matched Blueprints
* BDA creates a segment section each for each individual document that it has identified in the file. Each segment section has details on the matched blueprint and the results of the extraction. For each segment, BDA also outputs the page indices (one or more) from the original file.

* Get the custom output corresponding to each segment and look at the insights that BDA custom output produces.

In [ ]:
asset_id = 0
segments_metadata = next(item["segment_metadata"]
                                for item in job_metadata["output_metadata"] 
                                if item['asset_id'] == asset_id)

standard_outputs = [
    json.loads(helper_functions.read_s3_object(segment_metadata.get('standard_output_path')))for segment_metadata in segments_metadata]
custom_outputs = [json.loads(helper_functions.read_s3_object(segment_metadata.get('custom_output_path'))) if segment_metadata.get('custom_output_status') == 'MATCH' else None for segment_metadata in segments_metadata]

In [ ]:
custom_outputs_json = JSON(custom_outputs, root="custom_outputs", expanded=False)
custom_outputs_table = pd.DataFrame(helper_functions.get_summaries(custom_outputs)).fillna('')

display_functions.display_multiple(
    [
        display_functions.get_view(custom_outputs_table.style.hide(axis='index')),
        display_functions.get_view(custom_outputs_json)
    ], 
    ["Table View", "Raw JSON"])

### Step-7: Async Extraction for Detailed Medical Chart 
* Invoke Data Automation Async API for extraction of **detailed_medical_chart_inf.pdf**
* This step returns the invocation ARN
* Poll invocation status
* Validate Classification
* View Matched Blueprints, Output Summary

### Step-7.1: Extraction
* Invoke Data Automation Async API for extraction of **detailed_medical_chart_inf.pdf**
* This step returns the invocation ARN

In [ ]:
response = bda_runtime_client.invoke_data_automation_async(
    inputConfiguration={
        's3Uri': detailed_document_s3_uri
    },
    outputConfiguration={
        's3Uri': bda_s3_output_location
    },
    dataAutomationConfiguration={
        'dataAutomationProjectArn': project_arn,
        'stage': 'LIVE'
    }, 
    dataAutomationProfileArn = f'arn:aws:bedrock:{current_region}:{account_number}:data-automation-profile/us.data-automation-v1'
)

invocationArn = response['invocationArn']
print(f'Invoked data automation job with invocation arn {invocationArn}')

### Step-7.2: Poll for Invocation Status
* Since invocation is an async step, we will poll for status to be completed
* The invocation job status moves from `Created` to `InProgress` and finally to `Success` when the job completes successfully
* If the job encounters an error the final status is either `ServiceError` or `ClientError` with error details
* Run this step couple of time till the status is `Success`

In [ ]:
status_response = helper_functions.wait_for_completion(
            client=bda_client,
            get_status_function=bda_runtime_client.get_data_automation_status,
            status_kwargs={'invocationArn': invocationArn},
            completion_states=['Success'],
            error_states=['ClientError', 'ServiceError'],
            status_path_in_response='status',
            max_iterations=15,
            delay=30
)
if status_response['status'] == 'Success':
    job_metadata_s3_location = status_response['outputConfiguration']['s3Uri']
else:
    raise Exception(f'Invocation Job Error, error_type={status_response["error_type"]},error_message={status_response["error_message"]}')

### Step-7.3: Validate Classification
* Retrieve the job metadata. The Job metadata contains the S3 uri's for the standard output,custom output and the status of custom output. The custom output status could be either of `MATCH` or `NO_MATCH`.
* `MATCH` indicates BDA was able to find a matching blueprint for the specific segment from the list of blueprint we associated with the project.


In [ ]:
job_metadata = json.loads(helper_functions.read_s3_object(job_metadata_s3_location))

job_metadata_table = pd.DataFrame(job_metadata['output_metadata'][0]['segment_metadata']).fillna('')
job_metadata_table.index.name='Segment Index'
job_metadata_json = JSON(job_metadata, root="job_metadata", expanded=True)
# Display the widget
display_functions.display_multiple(
    [display_functions.get_view(job_metadata_table), display_functions.get_view(job_metadata_json)], 
    ["Table View", "Raw JSON"])

### Step-7.4: View  and Matched Blueprints

* BDA creates a segment section each for each individual document that it has identified in the file. Each segment section has details on the matched blueprint and the results of the extraction. For each segment, BDA also outputs the page indices (one or more) from the original file.
* Get the custom output corresponding to each segment and look at the insights that BDA custom output produces.

In [ ]:
asset_id = 0
segments_metadata = next(item["segment_metadata"]
                                for item in job_metadata["output_metadata"] 
                                if item['asset_id'] == asset_id)

standard_outputs = [
    json.loads(helper_functions.read_s3_object(segment_metadata.get('standard_output_path')))for segment_metadata in segments_metadata]
custom_outputs = [json.loads(helper_functions.read_s3_object(segment_metadata.get('custom_output_path'))) if segment_metadata.get('custom_output_status') == 'MATCH' else None for segment_metadata in segments_metadata]


In [ ]:
custom_outputs_json = JSON(custom_outputs, root="custom_outputs", expanded=False)
custom_outputs_table = pd.DataFrame(helper_functions.get_summaries(custom_outputs)).fillna('')

display_functions.display_multiple(
    [
        display_functions.get_view(custom_outputs_table.style.hide(axis='index')),
        display_functions.get_view(custom_outputs_json)
    ], 
    ["Table View", "Raw JSON"])

### Step-8: Upload formatted json to S3

* Format the output json with confidence scores and upload to S3 **idp-workshop-<account_id>-us-west-2/BDA-output**

In [ ]:
# Create simplified result
result = {}
explainability_info = custom_outputs[0]['explainability_info']

for field_name, field_data in explainability_info[0].items():
    if isinstance(field_data, dict) and 'confidence' in field_data and 'value' in field_data:
        result[field_name] = {
            "confidence": field_data['confidence'],
            "value": field_data['value']
        }

# Convert result to JSON string
result_json = json.dumps(result, indent=2)
print(result_json)

s3_client.put_object(
    Bucket=bucket_name,
    Key=f"{output_folder}/{output_file_name}",
    Body=result_json,
    ContentType='application/json'
)